# Maturity Report

End-to-end runner for the maturity suite. Runs every pair the
maturity tests exercise and prints the actual sub-scores so
thresholds can be calibrated and prose can quote the right numbers.

Run all cells top-to-bottom.

## Setup

In [ ]:
# Standard imports + repo path wiring. We add both the repo root and the
# `model_evaluation/` flat-layout module dir so the bpmn_* modules import
# cleanly when the notebook is opened from anywhere under the repo.

from __future__ import annotations

import sys
import json
from pathlib import Path
from typing import Optional

import pandas as pd

# Resolve repo root from this notebook's location. The notebook lives in
# `notebooks/` — go up one level.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pyproject.toml").exists(), f"could not find repo root from {Path.cwd()}"

MODEL_EVAL = REPO_ROOT / "model_evaluation"
SCRIPTS_DIR = REPO_ROOT / "scripts"
for p in (str(REPO_ROOT), str(MODEL_EVAL), str(SCRIPTS_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

EXAMPLES = REPO_ROOT / "examples"
MATURITY_DIR = EXAMPLES / "maturity"

print(f"Repo root: {REPO_ROOT}")
print(f"Maturity fixtures: {MATURITY_DIR}")

In [2]:
# Pipeline imports. These are the same modules the dashboard and the
# pytest suite use; the notebook is just a different orchestrator on top.

from BPMN_conversion import BPMNConverter, XMLBPMNConverter
from bpmn_similarity import (
    calculate_bpmn_similarity,
    calculate_trace_similarity,
)
from trace_extraction import extract_traces


def load_model(path: Path):
    # Mirror comparison_widget._load_model: XML/BPMN through the XML
    # converter, anything else as Signavio JSON.
    if path.suffix in (".xml", ".bpmn"):
        return XMLBPMNConverter.convert_file(str(path)).to_dict()
    with path.open("r", encoding="utf-8") as f:
        return BPMNConverter.convert(json.load(f)).to_dict()

## Edge-case pair scores

Every pair the maturity test suite exercises, with the actual
sub-scores measured under the current converter, normalizer, and
embedding model. The notebook does *not* assert thresholds here — it
just prints them, so you can see at a glance whether the pytest
assertions still have a healthy margin or whether something has
drifted.

In [ ]:
# (category, left_relpath_under_examples/maturity, right_relpath, note)
EDGE_PAIRS = [
    ("sanity", "sanity/identical_baseline.bpmn", "sanity/identical_baseline.bpmn",
     "identical — expect score near the top of the range everywhere"),
    ("sanity", "sanity/disjoint_left_credit.bpmn", "sanity/disjoint_right_student.bpmn",
     "disjoint — expect low overall"),
    ("sanity", "sanity/identical_baseline.bpmn", "sanity/renamed_only_b.bpmn",
     "renamed only — raw modest, normalized should lift substantially"),
    ("gateway", "gateway_substitutions/gateway_and.bpmn", "gateway_substitutions/gateway_xor.bpmn",
     "different gateway type, shared domain"),
    ("gateway", "gateway_substitutions/gateway_and.bpmn", "gateway_substitutions/gateway_or.bpmn",
     "different gateway type, shared domain"),
    ("gateway", "gateway_substitutions/gateway_xor.bpmn", "gateway_substitutions/gateway_or.bpmn",
     "different gateway type, shared domain"),
    ("structural", "structural_perturbations/linear_baseline.bpmn",
     "structural_perturbations/linear_reorder.bpmn",
     "small perturbation on linear sequence"),
    ("structural", "structural_perturbations/linear_baseline.bpmn",
     "structural_perturbations/linear_drift.bpmn",
     "larger perturbation on linear sequence"),
    ("structural", "structural_perturbations/and_two_branches.bpmn",
     "structural_perturbations/and_three_branches.bpmn",
     "branch added"),
    ("semantic", "semantic_naming/paraphrase_a.bpmn", "semantic_naming/paraphrase_b.bpmn",
     "paraphrase — relies on normalization"),
    ("semantic", "semantic_naming/synonym_a.bpmn", "semantic_naming/synonym_b.bpmn",
     "synonym — relies on normalization"),
    ("subprocess", "subprocess_folding/flat.bpmn", "subprocess_folding/with_subprocess.bpmn",
     "flat vs expanded subprocess — same labels"),
    ("round_trip", "format_round_trip/linear_sequence.bpmn",
     "format_round_trip/linear_sequence.json",
     "BPMN vs Signavio JSON"),
    ("round_trip", "format_round_trip/credit.bpmn", "format_round_trip/credit.json",
     "BPMN vs Signavio JSON"),
    ("degenerate", "degenerate/unsound_and_no_join.bpmn",
     "degenerate/sound_and_with_join.bpmn",
     "unsound vs sound — must not raise"),
    ("degenerate", "degenerate/empty.bpmn", "degenerate/empty.bpmn",
     "empty self — finite, no crash"),
    ("degenerate", "degenerate/single_task.bpmn", "degenerate/single_task.bpmn",
     "single-task self — expect a perfect self-comparison"),
    ("degenerate", "degenerate/empty.bpmn", "degenerate/single_task.bpmn",
     "empty vs single-task — finite"),
]


def safe_trace_similarity(model_a, model_b) -> Optional[float]:
    try:
        res_a = extract_traces(model_a, timeout_seconds=10.0, max_loop_depth=3)
        res_b = extract_traces(model_b, timeout_seconds=10.0, max_loop_depth=3)
        return calculate_trace_similarity(res_a, res_b, method="jaccard")
    except Exception:  # surfaced via the cell output, no need to crash the notebook
        return None

In [10]:
rows = []
for category, left_name, right_name, note in EDGE_PAIRS:
    left_path = MATURITY_DIR / left_name
    right_path = MATURITY_DIR / right_name
    if not left_path.exists() or not right_path.exists():
        rows.append({
            "category": category, "left": left_name, "right": right_name,
            "overall": None, "elements": None, "trace": None,
            "note": f"MISSING — {note}",
        })
        continue
    left = load_model(left_path)
    right = load_model(right_path)
    struct = calculate_bpmn_similarity(left, right, method="dice")
    rows.append({
        "category": category,
        "left": left_name,
        "right": right_name,
        "overall": struct["overall"],
        "elements": struct["high_level_scores"].get("elements"),
        "trace": safe_trace_similarity(left, right),
        "note": note,
    })

edge_df = pd.DataFrame(rows)
# Display with rounded scores for readability.
edge_df_display = edge_df.copy()
for col in ("overall", "elements", "trace"):
    edge_df_display[col] = edge_df_display[col].apply(
        lambda v: f"{v:.3f}" if isinstance(v, float) else v
    )
edge_df_display

Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 34561 partial trace(s); 35047 loop-cap hit(s)


Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 30 partial trace(s); 1 distinct deadlock marking(s)


Trace extraction recovered partial results for net '<unnamed>': 0 sound variant(s), 4 partial trace(s); 1 distinct deadlock marking(s)


,category,left,right,overall,elements,trace,note
0,sanity,sanity/identical_baseline.bpmn,sanity/identical_baseline.bpmn,1.000,1.000,1.000,identical — expect ~1.0 everywhere
1,sanity,sanity/disjoint_left_credit.bpmn,sanity/disjoint_right_student.bpmn,0.087,0.291,0.000,disjoint — expect low overall
2,sanity,sanity/renamed_only_a.bpmn,sanity/renamed_only_b.bpmn,0.281,0.750,0.000,"renamed only — raw modest, normalized should jump"
3,gateway,gateway_substitutions/gateway_and.bpmn,gateway_substitutions/gateway_xor.bpmn,0.325,0.500,0.000,"different gateway type, shared domain"
4,gateway,gateway_substitutions/gateway_and.bpmn,gateway_substitutions/gateway_or.bpmn,0.413,0.708,0.000,"different gateway type, shared domain"
5,gateway,gateway_substitutions/gateway_xor.bpmn,gateway_substitutions/gateway_or.bpmn,0.325,0.500,0.000,"different gateway type, shared domain"
6,structural,structural_perturbations/linear_baseline.bpmn,structural_perturbations/linear_reorder.bpmn,0.268,0.714,0.000,small perturbation on linear sequence
7,structural,structural_perturbations/linear_baseline.bpmn,structural_perturbations/linear_drift.bpmn,0.174,0.464,0.000,larger perturbation on linear sequence
8,structural,structural_perturbations/and_two_branches.bpmn,structural_perturbations/and_three_branches.bpmn,0.597,0.822,0.000,branch added
9,semantic,semantic_naming/paraphrase_a.bpmn,semantic_naming/paraphrase_b.bpmn,0.281,0.750,0.000,paraphrase — relies on normalization


### Normalization on the renamed pair

The "renamed only" claim is the canonical maturity story: same shape,
different label strings. Raw similarity should be modest; after
`normalize_atomic_names` aligns the second model's vocabulary to the
first's, the score should jump close to 1.0.

In [ ]:
from bpmn_normalization import normalize_atomic_names
from utils.string_similarity import cosine_sim_optimized

renamed_pairs = [
    ("renamed_only",
     MATURITY_DIR / "sanity" / "identical_baseline.bpmn",
     MATURITY_DIR / "sanity" / "renamed_only_b.bpmn"),
    ("paraphrase",
     MATURITY_DIR / "semantic_naming" / "paraphrase_a.bpmn",
     MATURITY_DIR / "semantic_naming" / "paraphrase_b.bpmn"),
    ("synonym",
     MATURITY_DIR / "semantic_naming" / "synonym_a.bpmn",
     MATURITY_DIR / "semantic_naming" / "synonym_b.bpmn"),
    ("disjoint",
     MATURITY_DIR / "sanity" / "disjoint_left_credit.bpmn",
     MATURITY_DIR / "sanity" / "disjoint_right_student.bpmn"),
]

rows = []
for name, a_path, b_path in renamed_pairs:
    a = load_model(a_path)
    b = load_model(b_path)
    raw = calculate_bpmn_similarity(a, b, method="dice")["overall"]
    aligned, _ = normalize_atomic_names(a, b, cosine_sim_optimized, threshold=0.7)
    norm = calculate_bpmn_similarity(a, aligned, method="dice")["overall"]
    rows.append({"pair": name, "raw": f"{raw:.3f}", "normalized": f"{norm:.3f}"})

pd.DataFrame(rows)

The expected pattern: `renamed_only`, `paraphrase`, and `synonym` should
all sit at a similar modest raw score and lift close to a perfect score
after normalization; `disjoint` should stay low both before and after.
That's the evidence behind the paper's maturity claim about semantic
normalization.